**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Compressed Sensing

Break Nyquist — legally. If a signal is *sparse* in some basis, you can reconstruct it from far fewer measurements than samples, by asking the right (random!) questions and solving the right (L1!) puzzle. Two sessions: why it works, and a full reconstruction you'll code yourself. This is the mathematics behind fast MRI.

## 1. Pre-requisites

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S1–S2 (bases, least squares).
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S1 (convexity).
- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (sampling — the law we're bending).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Why Undersampling Can Work* (~35 min)
**Goal:** sparsity + incoherent measurements + the L1 trick: the three-ingredient recipe.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (reconstruction lab).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Why Undersampling Can Work</b></summary>

**Timing (~35 min).** 10 min what Nyquist actually promises · 8 min sparsity and incoherence · 12 min the L1-versus-L2 geometry · 5 min buffer.

**Open by defending Nyquist, not attacking it.** The title says "break Nyquist," and students will hear that as "Nyquist was wrong." It was not. Nyquist is a *worst-case* guarantee: it protects you against every bandlimited signal, including the adversarial ones. Compressed sensing does not violate it — it changes the hypothesis, assuming the signal is sparse in some basis, and worst-case bounds are always beatable once you assume more. Getting this straight in the first two minutes prevents a whole category of confusion.

**Board first — the three ingredients, written as a list you keep returning to.** Sparsity (few coefficients matter), incoherence (each measurement touches a little of everything), L1 (the convex surrogate that finds sparse answers). Every result in the workshop is one of these three doing its job, and the failure experiments at the end are each of them being removed in turn.

**Make incoherence concrete with a bad example.** Ask what happens if you measure the *first* 150 time samples instead of 150 random ones. You learn a great deal about the start of the signal and nothing about the rest — the measurements are redundant with one another. Random sampling means every measurement is a fresh, near-independent question about all coefficients at once. Then note the pairing in the lab: random time samples against a DCT basis is close to maximally incoherent, which is exactly why MRI (Fourier measurements, wavelet-sparse images) works so well.

**The geometry is the session — do it at the board before the plot.** Draw a line (the solution set of $y = \Phi x$) and inflate a circle from the origin until it touches. It touches at a generic point, both coordinates nonzero. Now inflate a diamond. Its corners stick out along the axes, so unless the line is exactly parallel to a face, it hits a *corner* first — and corners are the points where a coordinate is zero. Sparsity is not imposed by the algorithm; it falls out of the shape of the ball. If students take one image from this workshop, make it this one.

**Ask the room.** "Why not just minimise $\|x\|_0$ directly — count the nonzeros?" Because it is combinatorial: you would have to check every possible support, $\binom{N}{K}$ of them, which for $N = 1024$, $K = 5$ is about $10^{12}$. L1 is the convex relaxation that is actually solvable, and the deep theorem (Candès–Romberg–Tao, Donoho) says that under incoherence it gives the *same answer*. State it as stated-not-proved; the notebook is honest about that and so should you.

**Pacing note.** The plotting code in the demo cell is fiddly and not worth reading aloud — project the figure, talk about the picture. If someone asks, the L2 panel draws the ball at the radius where it first meets the line, which is why the code computes that radius explicitly.
</details>

## 2. The Three Ingredients

💡 **Intuition.** Nyquist protects you against the *worst-case* signal. But natural signals aren't worst-case — they're **sparse**: a few active coefficients in the right basis ([Linear Algebra S1](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)'s smooth bump needed ~6 cosine coefficients). If only $K \ll N$ numbers matter, shouldn't $\sim K \log N$ *well-chosen* measurements suffice? They do, if the measurements are **incoherent** — each one touches a little of everything (random projections are perfect) — so that few measurements still 'see' every possible sparse pattern.

**The recovery puzzle.** Measurements $y = \Phi x$ with $\Phi \in \mathbb{R}^{m\times N}$, $m \ll N$: infinitely many $x$ fit. Choosing the sparsest ($\min \|x\|_0$) is combinatorial. The miracle: minimizing the **L1 norm** — convex! — finds the same answer under incoherence conditions (RIP; Candès–Romberg–Tao / Donoho, stated not proved).

💡 **Intuition.** Why L1 and not L2? Geometry. The solution set of $y = \Phi x$ is a flat (affine subspace); recovery inflates a norm-ball until it first touches the flat. The L2 ball is round — it touches at a generic point with *all* coordinates nonzero. The L1 ball is a diamond whose **corners sit on the axes** — flats almost always touch a corner first, and corners are sparse points. Sparsity falls out of the shape of the ball.

In [2]:
# The diamond-vs-ball picture, in 2-D
th = np.linspace(0, 2*np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
# constraint line y = Φx: x0 + 2 x1 = 1.4
for ax, name in [(axes[0], "L2"), (axes[1], "L1")]:
    xs = np.linspace(-1.6, 1.6, 10)
    ax.plot(xs, (1.4 - xs)/2, "k", linewidth=1, label="all x with Φx = y")
    if name == "L2":
        r = 1.4/np.sqrt(5)
        ax.plot(r*np.cos(th)*np.sqrt(5)/np.sqrt(5), r*np.sin(th), "C0")
        ax.plot(np.cos(th)*0.626, np.sin(th)*0.626, "C0")
        pt = np.array([1.4, 2.8])/5
    else:
        s = 0.7
        ax.plot([s,0,-s,0,s], [0,s,0,-s,0], "C1")
        pt = np.array([0, 0.7])
    ax.plot(*pt, "r*", markersize=14, label="first touch")
    ax.set_title(f"{name} ball meets the flat: {'dense point' if name=='L2' else 'CORNER → sparse!'}")
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.2, 1.4); ax.legend(fontsize=7); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2034816/541760034.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The same constraint line — every $x$ consistent with our measurement — meets two different norm balls, and where it touches decides everything. The L2 ball is round, so it makes contact at a generic point with **both** coordinates nonzero: a dense answer. The L1 ball is a diamond whose corners sit exactly on the axes, and the line hits a **corner**, where one coordinate is exactly zero: a sparse answer.

The crucial word is *exactly*. L2 shrinks coefficients toward zero but essentially never sets any to zero, because a round ball has no preferred directions. L1's corners are the preferred directions, and they are precisely the sparse points. Sparsity is therefore not something the algorithm searches for or a penalty we tune until it appears — it is a consequence of the *shape* of the ball being inflated.

**Why this generalises past two dimensions.** In $N$ dimensions the L1 ball is a cross-polytope with $2N$ vertices on the axes, plus edges and faces of every intermediate dimension — and all of its low-dimensional faces are exactly the sparse sets. A random affine subspace of dimension $N - m$ overwhelmingly tends to meet such a body at a low-dimensional face rather than in the middle of a high-dimensional one. That is the geometric content of the recovery theorems, and it is why the picture is worth trusting rather than treating as a two-dimensional coincidence.

The same figure explains the shrinkage step in Session 2's solver. The soft-threshold $\mathrm{sign}(u)\max(|u|-\tau, 0)$ is what the L1 ball's geometry becomes when you write it as an algorithm: move toward zero, and if you would overshoot, *stop at exactly zero*. That flat region at zero is the corner, in one dimension.

Keep in mind what the picture assumes. The line here was drawn at a generic angle. A constraint set aligned with a face of the diamond rather than pointed at a corner would recover a dense answer, and that is precisely the situation coherent measurements produce — which is why the sampling has to be random, and why the notebook's final experiment removes the randomness to show the whole thing collapse.

---
### 🕐 Session 2 of 2 — *Reconstruction Lab* (~40 min)
**Goal:** recover a sparse spectrum from 15% of the samples with ISTA, coded from scratch.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Reconstruction Lab</b></summary>

**Timing (~40 min).** 10 min the setup and ISTA · 10 min the recovery result · 8 min the L2 control · 12 min the three knob-turning experiments, which are the real lab.

**ISTA is two things students already know, glued.** A gradient step on the smooth least-squares part, then a soft-threshold. Write the update and label the halves: `c - eta * AT(A(c) - y)` is ordinary [gradient descent](../Intro_Math/Optimization/Optimization.ipynb), and `sign(c) * maximum(abs(c) - lam*eta, 0)` is the L1 ball's corner geometry expressed as an operation — shrink toward zero, and if you would cross zero, stop *at* zero. Nobody needs proximal operator theory to see that; the flat region at zero is the corner from Session 1, in one dimension.

**Point at `A` and `AT` as functions, not matrices.** The measurement operator is never formed as a $150 \times 1024$ matrix — `A` is "inverse DCT, then keep 150 entries" and `AT` is "scatter into a zero vector, then DCT." This is how it is done at scale: MRI reconstructions apply operators via FFTs and never materialise anything. Worth thirty seconds, because it is the difference between a toy and a technique. (The L2 control cell *does* build the explicit matrix, purely because `lstsq` needs one.)

**Set up the support check as the real test.** Before running, say what success would look like: not "the SNR is high" but "the recovered indices are the true indices." Getting all five of 1024 positions exactly right is a combinatorial statement — there are about $10^{12}$ possible supports — and it is far more convincing than any decibel figure. Then reveal that both happen.

**The L2 control is the cell that proves the point; do not skip it.** 0.8 dB against 25.8 dB, on *identical data* with an identical measurement operator. Only the recovery principle differs. This is a properly controlled experiment, and it is worth naming as one: students see plenty of demos where the good method also gets more data or more tuning. Here nothing changes but the norm.

**Ask the room.** "The L2 answer is the minimum-norm solution and it fits the 150 measurements exactly. So why is it wrong?" Because fitting the data was never the difficulty — infinitely many signals fit. Undersampled recovery is entirely about which of those you pick, and L2's round ball picks a dense one that spreads energy across all 1024 coefficients. The measurements do not distinguish them; the *prior* does.

**Run at least one knob experiment live — the uniform-sampling one is the best.** Replace the random `keep` with `np.arange(0, N, N//m)` and recovery collapses. That failure is the most instructive moment available in the workshop, because it shows randomness is load-bearing rather than a convenience: uniform sampling is coherent with the DCT basis, aliases fold tones onto each other indistinguishably, and no solver can undo it. If time allows, also drop $m$ toward the $\sim 2K\log N \approx 70$ region and watch performance fall off a cliff rather than degrade gracefully — phase transitions are characteristic of this field.

**A caveat to state plainly.** The signal here is *exactly* sparse — five nonzero coefficients and 1019 exact zeros. Real signals are compressible, not sparse: coefficients decay but nothing is exactly zero. The theory covers that case too, with error bounds relative to the best $K$-term approximation, but the demo is the idealised version and students should know it.
</details>

## 3. The Lab

Scenario: a signal made of 5 tones. Nyquist says record all $N = 1024$ samples; we record **150 random ones** and recover the whole thing.

Solver: **ISTA** (iterative soft-thresholding) for the LASSO form $\min_c \tfrac12\|y - \Phi \Psi c\|^2 + \lambda \|c\|_1$ — just [gradient descent](../Intro_Math/Optimization/Optimization.ipynb) on the smooth part, plus a *shrink-toward-zero* step that enforces sparsity:
$$c \leftarrow \mathrm{soft}_{\lambda\eta}\big(c - \eta \, A^T(Ac - y)\big), \qquad \mathrm{soft}_\tau(u) = \mathrm{sign}(u)\max(|u| - \tau, 0)$$

In [3]:
N, m = 1024, 150
# sparse-in-frequency signal: 5 random tones (real DCT basis keeps everything real)
from scipy.fft import dct, idct
c_true = np.zeros(N)
support = rng.choice(N, 5, replace=False)
c_true[support] = rng.uniform(1, 3, 5) * rng.choice([-1, 1], 5)
x_true = idct(c_true, norm="ortho")

# measure: m random time samples (a random row-selector Φ — maximally incoherent with DCT)
keep = np.sort(rng.choice(N, m, replace=False))
y = x_true[keep]

def A(c):  return idct(c, norm="ortho")[keep]          # Φ Ψ c
def AT(r):
    z = np.zeros(N); z[keep] = r
    return dct(z, norm="ortho")                         # (Φ Ψ)ᵀ r

# ISTA
c = np.zeros(N); eta, lam = 1.0, 0.02
for it in range(400):
    c = c - eta * AT(A(c) - y)
    c = np.sign(c) * np.maximum(np.abs(c) - lam * eta, 0)

x_rec = idct(c, norm="ortho")
print(f"recovered support: {np.sort(np.abs(c).argsort()[-5:])}")
print(f"true support:      {np.sort(support)}")
print(f"reconstruction SNR: {10*np.log10(np.var(x_true)/np.var(x_rec - x_true)):.1f} dB from {m}/{N} = {m/N:.0%} of samples")

recovered support: [275 315 522 650 867]
true support:      [275 315 522 650 867]
reconstruction SNR: 25.8 dB from 150/1024 = 15% of samples


**What just happened.** The recovered support is `[275 315 522 650 867]` and the true support is `[275 315 522 650 867]` — **identical**. Plus 25.8 dB reconstruction SNR from 150 of 1024 samples, 15% of what Nyquist would demand.

The support match is the stronger of the two results, and it is worth saying why. The SNR is a continuous score that a merely decent answer could earn. Identifying *which* five of 1024 coefficients are active is combinatorial: there are $\binom{1024}{5} \approx 10^{12}$ possible supports, and the algorithm found the right one from 150 numbers. That is not "approximately recovered" — it is exact recovery of the discrete structure, with the amplitudes then following easily.

**The budget, in the theory's own terms.** Recovery needs roughly $m \sim 2K\log N$ measurements, which here is about 70; we used 150, comfortably above. That is 30 measurements per active coefficient rather than the 205 samples per coefficient Nyquist would have required. The saving comes from asking better questions, not from taking more of them.

**What each ingredient contributed.** Sparsity: only 5 of 1024 DCT coefficients are nonzero, so there is little to determine. Incoherence: random *time* samples against a *DCT* basis is close to the maximally incoherent pairing — each measurement sees a little of every frequency, so no coefficient hides. L1: the soft-threshold in the loop is Session 1's corner geometry, driving small coefficients to exactly zero rather than merely shrinking them. Remove any one and the reconstruction fails, which is precisely what the closing experiments demonstrate.

**Read the 25.8 dB honestly.** It is not infinite, and this problem is noiseless — the only error comes from ISTA being a first-order method run for a fixed 400 iterations with a fixed $\lambda$. More iterations, or a decreasing $\lambda$, or a debiasing least-squares fit restricted to the recovered support, would push it substantially higher. The `lam = 0.02` that enforces sparsity also biases the surviving amplitudes downward, which is a known cost of L1 and the reason practitioners often re-fit on the discovered support afterwards.

**And note the idealisation.** This signal is *exactly* sparse: five nonzeros and 1019 exact zeros. Real signals are compressible rather than sparse — coefficients decay, but nothing is exactly zero. The theory handles that with error bounds relative to the best $K$-term approximation, so the technique survives, but the clean support-recovery result above is a property of the idealised setup.

In [4]:
fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
axes[0].plot(x_true, linewidth=1, label="true signal (1024 samples)")
axes[0].plot(keep, y, "r.", markersize=3, label="the 150 we measured")
axes[0].legend(fontsize=8); axes[0].set_xlim(0, 400)
axes[1].plot(x_rec, linewidth=1, color="C2", label="L1 reconstruction from 15%")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2034816/3084294164.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The top panel shows the true signal with the 150 measured samples marked in red — sparse, irregular dots scattered across a waveform that oscillates far faster than they do. Anyone looking at those red dots alone would say they are hopelessly undersampled, and by Nyquist's standard they are. The bottom panel is the reconstruction from exactly those dots, and it reproduces the full 1024-sample waveform.

**The irregularity of the red dots is the point, not an aesthetic choice.** Uniform sampling at 150 points would alias every tone above the resulting Nyquist limit down onto lower frequencies, and those aliases are *indistinguishable* — no algorithm can undo a genuine ambiguity. Random sampling replaces coherent aliasing with incoherent, noise-like leakage spread thinly across all frequencies. That leakage is small and unstructured, so a sparse prior can rise above it, whereas a clean alias would be a competing explanation of equal quality. This is the single most important practical takeaway of the workshop, and it is why the closing experiment that switches to uniform sampling breaks everything.

Worth noting that the plot shows only the first 400 samples for legibility; the reconstruction covers all 1024, and the 25.8 dB figure is computed over the whole signal rather than the visible window.

If you compare the two panels closely you will find small amplitude discrepancies rather than structural errors — the tones are in the right places with slightly conservative heights. That is the L1 penalty's known bias, shrinking every surviving coefficient a little, and it is what a debiasing re-fit on the recovered support would remove.

In [5]:
# Control experiment: least squares (L2) from the same 150 samples — total failure
# minimum-norm solution: c = Aᵀ(AAᵀ)⁻¹y, computed via lstsq on the explicit matrix
Psi_rows = idct(np.eye(N), norm="ortho", axis=0)[keep]     # the m×N matrix ΦΨ
c_l2, *_ = np.linalg.lstsq(Psi_rows, y, rcond=None)
x_l2 = idct(c_l2, norm="ortho")
print(f"L2 reconstruction SNR: {10*np.log10(np.var(x_true)/np.var(x_l2 - x_true)):.1f} dB   ← the round ball touching a dense point")
print(f"L1 reconstruction SNR: {10*np.log10(np.var(x_true)/np.var(x_rec - x_true)):.1f} dB   ← the diamond finding the corner")

L2 reconstruction SNR: 0.8 dB   ← the round ball touching a dense point
L1 reconstruction SNR: 25.8 dB   ← the diamond finding the corner


**What just happened.** **0.8 dB** for L2 against **25.8 dB** for L1 — a factor of about 316 in error power. And this is a genuinely controlled experiment: identical signal, identical 150 measurements, identical measurement operator. The *only* difference is which solution is selected from the infinitely many that fit.

That last point is the one to hold onto. The L2 answer is not a failed fit. `lstsq` returns the minimum-norm solution, which reproduces all 150 measurements exactly — it is perfectly consistent with everything we observed. Fitting the data was never the difficulty, because the system is underdetermined and infinitely many signals fit it exactly. Undersampled recovery is entirely a question of *which* consistent answer you choose, and that choice is made by the prior, not by the data.

**Why L2 chooses badly.** Minimising $\|c\|_2$ spreads energy as evenly as possible across all 1024 coefficients, since a round ball has no preferred directions — it is the least committal answer available. But the truth is maximally committal: five large coefficients and 1019 exact zeros. The minimum-energy solution is therefore about as far from the truth as a consistent answer can be, and 0.8 dB says the reconstruction carries barely more signal than error.

This is Session 1's geometry, measured. The round ball touches the constraint flat at a dense point; the diamond touches at a corner. Same flat, same measurements, opposite answers — and the 25-dB gap is what the shape of a norm ball is worth.

**The generalisable lesson.** When a problem is underdetermined, your regulariser is not a numerical convenience or a hyperparameter to tune — it *is* your model of what signals look like, and it decides the answer outright. Choosing L2 out of habit here would produce a result that fits every measurement and is nonetheless meaningless. The same reasoning governs ridge versus lasso in [estimation](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb), where L1 corresponds to a Laplacian prior and L2 to a Gaussian one.

**Knobs to turn** (5 minutes each): drop $m$ until recovery breaks (~$2K\log N$); make the signal less sparse (25 tones); sample *uniformly* instead of randomly and watch coherent aliasing kill it — randomness is not optional.

## 4. Conclusion

Sparsity + incoherent (random) measurements + L1's cornered geometry = signals from far fewer samples than Nyquist demands. The same three ingredients accelerate MRI scanners, single-pixel cameras, and radio astronomy.

---
## Where next

- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — wavelets: the sparsifying basis for images.
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — ISTA is proximal gradient descent; the theory generalizes.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — LASSO as MAP estimation with a Laplacian prior.